# Figures: gridded dataset

Makes the paper's figures and table on the gridded HySWE dataset: the validation against the independent NorSWE observations (Figs. 3 and 6, Table 6), and the comparison with ERA5-Land over the Northern Hemisphere and six GMBA mountain ranges (Figs. 7–9).


**Before running:** create `src/common/local_paths.py` from `local_paths.example.py` (this notebook only uses `BASE_PATH`), and start Jupyter from the `notebooks/` folder.

**Inputs**, relative to `BASE_PATH`, and what produces them. `<source>` is `hybrid` or `era5_land`; each script is run once per source with `--source <source>`.

| File | Produced by |
|---|---|
| `results/temp/norswe_valid_stations.csv` | `src/3_analysis/compute_metrics_norswe.py` (the same for both sources) |
| `results/temp/norswe_<source>_values_mt_<code>.csv` | `src/3_analysis/compute_metrics_norswe.py` |
| `results/temp/<source>_avg_peak_swe.tif` | `src/3_analysis/compute_avg_peak_swe_map.py` |
| `results/temp/<source>_<region>_doy_avg_swe.csv` | `src/3_analysis/compute_doy_swe.py` |
| `results/temp/<source>_northern_hemisphere_yearly_{peak_swe,css_length}.csv` | `src/3_analysis/compute_longterm_trends.py` |
| `data/GMBA_Inventory_v2.0_standard_300/GMBA_Inventory_v2.0_standard_300.shp` | the GMBA Mountain Inventory v2.0 |

Figure 3 also downloads the Natural Earth 1:50m land polygons for the base map.

The DOY and trend scripts average over the no-10s analysis mask (`data/analysis_mask_no10s.tif`), which leaves out the pixels where ERA5-Land holds its constant 10 m SWE. The NorSWE cells drop those values too (10000 mm).

Each section only needs *Setup* to have run, except *Table 6*, which uses the NorSWE values loaded for Figure 6.

## Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd
import rioxarray
import numpy as np
import math
import matplotlib.pyplot as plt

In [ ]:
# Paths: all files are read from the project folder BASE_PATH (data/ and results/),
# set per machine in src/common/local_paths.py (copy local_paths.example.py)
import sys
sys.path.append(str(Path.cwd().parent / 'src' / 'common'))
import local_paths

In [ ]:
# Define a style for the different data sources
style = {
    'Hybrid':    {'color': '#DAA520', 'linewidth': 2, 'label': 'HySWE'},  # Goldenrod Yellow
    'ERA5-Land': {'color': '#00C49F', 'linewidth': 2, 'label': 'ERA5-Land'}   # Bright Cyan-Green / Aquamarine
}

# Define the labels for the figure panels
panel_labels = [f"({letter})" for letter in "abcdefghijklmnopqrstuvwxyz"]

In [ ]:
# Define the codes for the different NorSWE measurement types
code_to_mt = {
    0: "Multi-point manual snow survey",
    1: "Single-point manual SWE measurement",
    2: "Snow pillow or snow scale",
    3: "Passive gamma",
    4: "GNSS/GPS methods",
    5: "Cosmic ray attenuation",
    6: "Time domain reflectometry",
    63: "Missing value",
    64: "Airborne gamma SWE",
    # 7–62 are reserved
}

## Figure 3: NorSWE stations used for the validation

The NorSWE stations left after removing the SNOTEL stations and those within 1′ of an NH-SWE station, coloured by measurement type (Equal Earth projection). The station list is written by `compute_metrics_norswe.py`.

In [ ]:
from pyproj import Transformer

# =============================================================================
# Configuration
# =============================================================================
plt.rcParams.update({"font.size": 12})

source_crs = "EPSG:4326"
target_crs = "EPSG:8857"  # Equal Earth

lon_min, lon_max = -180, 180
lat_min, lat_max = 0, 90

land_url = "https://naciscdn.org/naturalearth/50m/physical/ne_50m_land.zip"

gridline_color = "dimgray"
gridline_style = "--"
gridline_width = 0.4
gridline_alpha = 0.55

label_fontsize = 12
label_color = "dimgray"

boundary_color = "black"
boundary_linewidth = 0.8


# =============================================================================
# Load the valid NorSWE stations and transform their coordinates
# =============================================================================
stations_df = pd.read_csv(local_paths.BASE_PATH / 'results' / 'temp' / 'norswe_valid_stations.csv')

transformer = Transformer.from_crs(
    source_crs,
    target_crs,
    always_xy=True,
)

station_lon = stations_df["lon"].values
station_lat = stations_df["lat"].values
station_type = stations_df["type_mes"].values

station_in_domain = (
    (station_lon >= lon_min)
    & (station_lon <= lon_max)
    & (station_lat >= lat_min)
    & (station_lat <= lat_max)
)


# =============================================================================
# Read, clip, and project the land-vector basemap
# =============================================================================
land_gdf = gpd.read_file(land_url)

if land_gdf.crs is None:
    land_gdf = land_gdf.set_crs(source_crs)

land_gdf = gpd.clip(land_gdf, (lon_min, lat_min, lon_max, lat_max)).to_crs(target_crs)


# =============================================================================
# Create the displayed-domain boundary
# =============================================================================
lon_left = -180 + 1e-6
lon_right = 180 - 1e-6

n_boundary = 1_000

# Bottom edge: 0°N
bottom_lon = np.linspace(lon_left, lon_right, n_boundary)
bottom_lat = np.full_like(bottom_lon, lat_min)

# Eastern edge: 0–90°N at approximately 180°E
right_lat = np.linspace(lat_min, lat_max, n_boundary)
right_lon = np.full_like(right_lat, lon_right)

# Top edge: 90°N, from east to west
top_lon = np.linspace(lon_right, lon_left, n_boundary)
top_lat = np.full_like(top_lon, lat_max)

# Western edge: 90–0°N at approximately 180°W
left_lat = np.linspace(lat_max, lat_min, n_boundary)
left_lon = np.full_like(left_lat, lon_left)

boundary_segments = [
    (bottom_lon, bottom_lat),
    (right_lon, right_lat),
    (top_lon, top_lat),
    (left_lon, left_lat),
]

projected_segments = [
    transformer.transform(lon_segment, lat_segment)
    for lon_segment, lat_segment in boundary_segments
]

boundary_x = np.concatenate([
    segment_x for segment_x, segment_y in projected_segments
])

boundary_y = np.concatenate([
    segment_y for segment_x, segment_y in projected_segments
])

x_min, x_max = boundary_x.min(), boundary_x.max()
y_min, y_max = boundary_y.min(), boundary_y.max()


# =============================================================================
# Plot
# =============================================================================
fig, ax = plt.subplots(figsize=(18, 6), dpi=300)

# Land-vector base layer
land_gdf.plot(
    ax=ax,
    color="gainsboro",
    edgecolor="none",
    linewidth=0,
    zorder=0,
)


# =============================================================================
# Longitude / latitude graticules
# =============================================================================
meridians = np.arange(-135, 180, 45)
parallels = np.arange(20, 90, 20)

lat_line = np.linspace(lat_min, lat_max, 400)
lon_line = np.linspace(lon_min, lon_max, 720)

# Meridians
for lon in meridians:
    grid_x, grid_y = transformer.transform(
        np.full_like(lat_line, lon),
        lat_line,
    )

    ax.plot(
        grid_x,
        grid_y,
        color=gridline_color,
        linewidth=gridline_width,
        linestyle=gridline_style,
        alpha=gridline_alpha,
        zorder=1,
    )

    if lon == 0:
        label = "0°"
    elif lon < 0:
        label = f"{abs(lon):.0f}°W"
    else:
        label = f"{lon:.0f}°E"

    label_x, label_y = transformer.transform(lon, lat_min)

    ax.annotate(
        label,
        xy=(label_x, label_y),
        xytext=(0, -10),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=label_fontsize,
        color=label_color,
        clip_on=False,
        zorder=6,
    )

# Parallels
for lat in parallels:
    grid_x, grid_y = transformer.transform(
        lon_line,
        np.full_like(lon_line, lat),
    )

    ax.plot(
        grid_x,
        grid_y,
        color=gridline_color,
        linewidth=gridline_width,
        linestyle=gridline_style,
        alpha=gridline_alpha,
        zorder=1,
    )

    label_x, label_y = transformer.transform(lon_left, lat)

    ax.annotate(
        f"{lat:.0f}°N",
        xy=(label_x, label_y),
        xytext=(-10, 0),
        textcoords="offset points",
        ha="right",
        va="center",
        fontsize=label_fontsize,
        color=label_color,
        clip_on=False,
        zorder=6,
    )


# =============================================================================
# Stations, coloured by SWE measurement method
# =============================================================================
colours = [
    "#555555",  # Gray
    "#56B4E9",  # Blue
    "#E69F00",  # Orange
    "#009E73",  # Green
]

color_idx = 0
for code in code_to_mt.keys():
    method_mask = (
        station_in_domain
        & (station_type == code)
    )

    if not np.any(method_mask):
        continue

    station_x, station_y = transformer.transform(
        station_lon[method_mask],
        station_lat[method_mask],
    )

    ax.scatter(
        station_x,
        station_y,
        s=18,
        color=colours[color_idx],
        edgecolor="none",
        alpha=0.85,
        label=code_to_mt[code] + f" (n={np.sum(method_mask)})",
        zorder=3,
    )

    color_idx += 1


# =============================================================================
# Projected-domain outline
# =============================================================================
for segment_x, segment_y in projected_segments:
    ax.plot(
        segment_x,
        segment_y,
        color=boundary_color,
        linewidth=boundary_linewidth,
        solid_capstyle="butt",
        antialiased=True,
        zorder=5,
    )


# =============================================================================
# Legend and final formatting
# =============================================================================
legend = ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.02, 0.01),
    fontsize=16,
    frameon=False,
    markerscale=1.2,
)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect("equal")

# Graticule labels replace conventional Cartesian axes.
ax.set_axis_off()

fig.subplots_adjust(
    left=0.06,
    right=0.99,
    bottom=0.13,
    top=0.91,
)

plt.show()

In [ ]:
print(f"Total number of valid NorSWE stations (excluding SNOTEL and NH-SWE overlaps): {np.sum(station_in_domain)}")

## Figure 6: HySWE against the NorSWE observations

Hexbin plots per measurement type. The HySWE panels are Fig. 6; the cell draws the same panels for ERA5-Land, for comparison.

In [ ]:
# Sources
source_names = ['Hybrid', 'ERA5-Land']

mt_data = {source_name: {} for source_name in source_names}
for source_name in source_names:
    # Define the path to the temporary directory where the data is stored
    source_path_name = source_name.replace("-", "_").lower()
    temp_path = local_paths.BASE_PATH / 'results' / 'temp'
    if temp_path.exists():
        print(f'Retrieving {source_name} measurement files from: "{temp_path}"')
    else:
        raise FileNotFoundError(f"Temporary directory for {source_name} does not exist: {temp_path}")

    # Load the norswe values per type of measurement
    valid_types = set()
    for mt_code, mt_name in code_to_mt.items():
        path = temp_path / f'norswe_{source_path_name}_values_mt_{mt_code}.csv'
        if not path.exists():
            continue
        df = pd.read_csv(path)
        # Drop the ERA5-Land 10 m (10000 mm) values, as the no-10s analysis mask does for the maps
        n_10s = (df == 10000).sum().sum()
        if n_10s > 0:
            print(f"Setting {n_10s} ERA5-Land 10s values to NaN for measurement type {mt_code} ({mt_name}) for {source_name}.")
            df.replace(10000, np.nan, inplace=True)
        mt_data[source_name][mt_code] = df
        valid_types.add(mt_code)
    print(f"Valid measurement types found:\n{'\n'.join(f'{code} - {name}' for code, name in code_to_mt.items() if code in valid_types)}")

In [ ]:
# Increase the font size for the plot
plt.rcParams.update({"font.size": 24})

n_mt = len(mt_data[source_names[0]])

for source_name in source_names:
    source_path_name = source_name.replace("-", "_").lower()

    fig, axes = plt.subplots(
        1,
        n_mt,
        figsize=(8 * n_mt, 8),
        dpi=300,
        constrained_layout=True,
    )

    if n_mt == 1:
        axes = [axes]

    hexbins = []

    for panel_idx, (ax, (mt_code, df)) in enumerate(
        zip(axes, mt_data[source_name].items())
    ):

        # Drop rows with NaN values in either column to avoid issues with hexbin
        clean_df = df.dropna()
        x = clean_df["norswe"]
        y = clean_df[source_path_name]

        # Find the maximum value to set equal x/y limits
        max_val = max(
            np.percentile(clean_df["norswe"], 100),
            np.percentile(clean_df[source_path_name], 100),
        )

        hb = ax.hexbin(
            x,
            y,
            gridsize=50,
            cmap="viridis",
            extent=(0, max_val, 0, max_val),
            mincnt=1,
            bins="log",
        )
        hexbins.append(hb)

        ax.plot([0, max_val], [0, max_val], "r--", label="1:1 Line")

        if mt_code == 0:
            if source_name == "Hybrid":
                ax.set_ylabel("HySWE (mm)", fontsize=28)
            else:
                ax.set_ylabel(f"{source_name} SWE (mm)", fontsize=28)

        ax.set_title(code_to_mt[mt_code], pad=10)
        ax.set_xlim(0, max_val)
        ax.set_ylim(0, max_val)

        # Reduce the number of ticks on the axes for better readability
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))

        # Panel label
        ax.text(
            0.03,
            0.97,
            panel_labels[panel_idx],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=28,
            fontweight="bold",
            color="black",
            zorder=10,
        )

    # Synchronize the colour limits across all subplots
    vmin = min(h.get_clim()[0] for h in hexbins)
    vmax = max(h.get_clim()[1] for h in hexbins)

    for h in hexbins:
        h.set_clim(vmin, vmax)

    # Shared x-axis label
    fig.text(0.5, -0.05, "NorSWE (mm)", ha="center", va="center", fontsize=28)

    # Shared colourbar
    fig.colorbar(hexbins[0], ax=axes, label="Count", pad=0.01)

    # Add a legend for the 1:1 line
    axes[3].legend(loc="upper right", fontsize=24)

    plt.show()

## Table 6: NorSWE validation metrics

MAE, RMSE, bias and NSE per measurement type over the points with both sources (Table 6), followed by the RMSE reduction from ERA5-Land to HySWE and the share of the squared bias in the MSE quoted in Sect. 3.2.

In [ ]:
# Join both sources in a single df per measurement type
mt_data_joined = {mt_code: pd.DataFrame() for mt_code in mt_data[source_names[0]].keys()}
for source_name in source_names:
    for mt_code, df in mt_data[source_name].items():
        for col in df.columns:
            if col not in mt_data_joined[mt_code].columns:
                mt_data_joined[mt_code][col] = df[col]
            else:
                if not df[col].equals(mt_data_joined[mt_code][col]):
                    raise ValueError(f"Column '{col}' in source '{source_name}' for measurement type '{mt_code}' does not match the existing data.")

# Clean the joined dataframes by dropping rows with NaN values
for mt_code, df in mt_data_joined.items():
    mt_data_joined[mt_code] = df.dropna()

# For both sources and all measurement types, compare different metrics (e.g., RMSE, bias, correlation) and visualize them in a summary table or bar chart.
for source_name in source_names:
    source_path_name = source_name.replace("-", "_").lower()
    metrics = []
    for mt_code, df in mt_data_joined.items():

        # Calculate metrics
        mae = np.mean(np.abs(df['norswe'] - df[source_path_name]))
        rmse = np.sqrt(np.mean((df['norswe'] - df[source_path_name]) ** 2))
        bias = np.mean(df[source_path_name] - df['norswe'])
        correlation = np.corrcoef(df['norswe'], df[source_path_name])[0, 1]
        nse = 1 - (np.sum((df[source_path_name] - df['norswe']) ** 2) / np.sum((df['norswe'] - np.mean(df['norswe'])) ** 2))
        metrics.append({
            'Measurement Type': code_to_mt[mt_code],
            'MAE': mae,
            'RMSE': rmse,
            'Bias': bias,
            'NSE': nse,
            # 'Correlation': correlation
        })

    metrics_df = pd.DataFrame(metrics)
    print(f"\nMetrics Summary for {source_name}:\n")
    print(metrics_df)
    print()


In [ ]:
# Calculate the relative improvement in RMSE from ERA5-Land to Hybrid for each measurement type
rmse_relative_improvement = []
for mt_code in mt_data_joined.keys():
    df = mt_data_joined[mt_code]
    hybrid_rmse = np.sqrt(np.mean((df['norswe'] - df['hybrid']) ** 2))
    era5_rmse = np.sqrt(np.mean((df['norswe'] - df['era5_land']) ** 2))
    relative_improvement = (era5_rmse - hybrid_rmse) / era5_rmse * 100
    rmse_relative_improvement.append({
        'Measurement Type': code_to_mt[mt_code],
        'Hybrid RMSE': hybrid_rmse,
        'ERA5-Land RMSE': era5_rmse,
        'Relative Improvement (%)': relative_improvement
    })

pd.DataFrame(rmse_relative_improvement)


In [ ]:
# Calculate the percentage of bias with respect to the total error variance for each measurement type and source
bias_percentage = {}
for source_name in source_names:
    print(f"\nCalculating bias percentage for source: {source_name}")
    source_path_name = source_name.replace("-", "_").lower()
    bias_percentage[source_name] = {}
    for mt_code, df in mt_data_joined.items():
        bias_squared = np.mean(df[source_path_name] - df['norswe']) ** 2
        mse = np.mean((df[source_path_name] - df['norswe']) ** 2)
        bias_percentage[source_name][mt_code] = (bias_squared / mse) * 100
        print(f"Measurement Type: {code_to_mt[mt_code]}, Bias Percentage: {bias_percentage[source_name][mt_code]:.2f}%")

## Figure 7: average peak SWE maps

The HySWE peak SWE averaged over the 40 years, for the Northern Hemisphere and six mountain ranges.

In [ ]:
source = 'Hybrid'   # 'Hybrid', 'ERA5-Land'

In [ ]:
# Load the average peak SWE map for the specified source
source_path_name = source.lower().replace('-', '_')
filename = f'{source_path_name}_avg_peak_swe.tif'
avg_peak_swe_file = local_paths.BASE_PATH / 'results' / 'temp' / filename
with rioxarray.open_rasterio(avg_peak_swe_file) as avg_peak_swe_ds:
    avg_peak_swe = avg_peak_swe_ds.squeeze(dim='band', drop=True)

# Load the mountain masks
mm_filename = 'GMBA_Inventory_v2.0_standard_300'
mountain_masks_file = local_paths.BASE_PATH / 'data' / mm_filename / f'{mm_filename}.shp'
mountain_mask_gdf = gpd.read_file(mountain_masks_file)

In [ ]:
sub_regions = 'Pyrenees', 'European Alps', 'Sierra Nevada', 'Canadian Rockies', 'Eastern Sayan', 'Himalaya' # 'Pyrenees', 'European Alps', 'Sierra Nevada', 'Canadian Rockies', 'Coast Mountains', 'Caucasus Mountains', 'Eastern Sayan', 'Pamir Mountains', 'Southern Scandes', 'Himalaya'

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

show_nh = True
show_contours = True  # Now controls the crop-box overlays in the NH panel and outlines in subplots
n = 5
n_sub = len(sub_regions)

# Set the font size for the plots
plt.rcParams.update({'font.size': 12})

# Create a function to reduce the number of ticks on the map
def reduce_map_ticks(ax, nbins_x=3, nbins_y=3):
    """Limit the number of longitude and latitude ticks."""
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins_x, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins_y, integer=True))

# --- Panel letters ---
# NH panel is (a) when present; otherwise subplots start at (a)

if show_nh:
    top_panel_label = panel_labels[0]
    region_panel_labels = {
        region_name: panel_labels[i + 1]
        for i, region_name in enumerate(sub_regions)
    }
else:
    region_panel_labels = {
        region_name: panel_labels[i]
        for i, region_name in enumerate(sub_regions)
    }

# --- Layout calculations ---
sub_rows = 2 if n_sub > 5 else 1
sub_cols = math.ceil(n_sub / sub_rows)

def expand_bounds_to_match_axes(bounds, ax, fig):
    """Pad bounds so their lon/lat ratio matches the physical axes ratio."""
    minx, miny, maxx, maxy = bounds
    lon_span = maxx - minx
    lat_span = maxy - miny
    center_x = (minx + maxx) / 2
    center_y = (miny + maxy) / 2

    bbox = ax.get_position()
    fig_w, fig_h = fig.get_size_inches()
    box_w_in = bbox.width * fig_w
    box_h_in = bbox.height * fig_h
    target_ratio = box_w_in / box_h_in

    current_ratio = lon_span / lat_span

    if current_ratio < target_ratio:
        new_lon_span = lat_span * target_ratio
        minx = center_x - new_lon_span / 2
        maxx = center_x + new_lon_span / 2
    else:
        new_lat_span = lon_span / target_ratio
        miny = center_y - new_lat_span / 2
        maxy = center_y + new_lat_span / 2

    return minx, miny, maxx, maxy


# --- Prepare top (NH) data ---
map_coarse = (
    avg_peak_swe.coarsen(x=n, y=n, boundary="trim").mean()
    if show_nh else None
)

# --- Figure layout ---
n_rows = (1 if show_nh else 0) + sub_rows
fig_height = (3.5 if show_nh else 0) + (sub_rows * 4.5)

fig = plt.figure(figsize=(18, fig_height), dpi=250)
gs = GridSpec(
    n_rows,
    sub_cols + 1,
    figure=fig,
    height_ratios=[1] * n_rows,
    width_ratios=[1] * sub_cols + [0.04],
    wspace=0.15,
    hspace=0.25,
)

mappable = None

# --- Create top axes ---
if show_nh:
    ax_top = fig.add_subplot(gs[0, :sub_cols])

# --- Bottom subplots: calculate and store actual crop bounds ---
sub_start_row = 1 if show_nh else 0
plot_data = {}
expanded_bounds_by_region = {}
axes_bottom = []

for i, region_name in enumerate(sub_regions):
    r = sub_start_row + (i // sub_cols)
    c = i % sub_cols

    ax = fig.add_subplot(gs[r, c])
    axes_bottom.append(ax)

    orig_bounds = (
        mountain_mask_gdf[
            mountain_mask_gdf["MapName"] == region_name
        ].total_bounds
    )

    # This is the exact geographic extent displayed in the corresponding subplot
    expanded_bounds = expand_bounds_to_match_axes(orig_bounds, ax, fig)
    expanded_bounds_by_region[region_name] = expanded_bounds

    cropped = avg_peak_swe.rio.clip_box(*expanded_bounds)
    plot_data[region_name] = cropped

# --- Shared colour scale ---
all_mins = [ds.min().values for ds in plot_data.values()]
all_maxs = [ds.max().values for ds in plot_data.values()]

if show_nh:
    all_mins.append(map_coarse.min().values)
    all_maxs.append(map_coarse.max().values)

min_swe = np.nanmin(all_mins)
max_swe = np.nanmax(all_maxs)

# --- Top NH plot with sub-region crop boxes ---
if show_nh:
    p = map_coarse.plot(
        ax=ax_top,
        cmap="Blues",
        vmin=min_swe,
        vmax=max_swe,
        add_colorbar=False,
    )
    mappable = p

    # Label for the NH panel
    ax_top.text(
        0.01, 0.98,
        top_panel_label,
        transform=ax_top.transAxes,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="bold",
        zorder=20,
    )

    # Draw the exact crop extent of each regional subplot
    if show_contours:
        for region_name, (minx, miny, maxx, maxy) in expanded_bounds_by_region.items():

            box = Rectangle(
                (minx, miny),
                maxx - minx,
                maxy - miny,
                fill=False,
                edgecolor="k",
                linewidth=0.8,
                linestyle="--",
                zorder=10,
            )
            ax_top.add_patch(box)

            # Label the box with its corresponding subplot letter
            lon_padding = 0.02 * (maxx - minx)
            lat_padding = 0.04 * (maxy - miny)

            ax_top.text(
                minx + lon_padding,
                maxy + lat_padding,
                region_panel_labels[region_name],
                ha="left",
                va="bottom",
                fontsize=10,
                fontweight="bold",
                color="k",
                zorder=11,
            )

    ax_top.set_xlabel("")
    ax_top.set_ylabel("")
    ax_top.set_title("Northern Hemisphere")
    ax_top.set_xlim(-180, 180)
    ax_top.set_ylim(20, 84)
    # reduce_map_ticks(ax_top, nbins_x=5, nbins_y=4)
    ax_top.set_facecolor("lightgray")
    ax_top.set_aspect("equal")

# --- Sub-region panels ---
for i, region_name in enumerate(sub_regions):
    ax = axes_bottom[i]
    cropped = plot_data[region_name]

    p = cropped.plot(
        ax=ax,
        cmap="Blues",
        vmin=min_swe,
        vmax=max_swe,
        add_colorbar=False,
    )
    mappable = p

    if show_contours:
        region_geom = mountain_mask_gdf[
            mountain_mask_gdf["MapName"] == region_name
        ]
        region_geom.plot(
            ax=ax,
            facecolor="none",
            edgecolor="k",
            linewidth=0.5,
            linestyle="--",
        )

    # Panel letter, corresponding to the label in the NH crop box
    ax.text(
        0.01, 0.98,
        region_panel_labels[region_name],
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=16,
        fontweight="bold",
        zorder=20,
    )

    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_title(region_name)
    ax.set_facecolor("lightgray")
    ax.set_aspect("equal")
    reduce_map_ticks(ax, nbins_x=5, nbins_y=4)
# --- Shared colourbar and labels ---
cax = fig.add_subplot(gs[:, sub_cols])
cbar = fig.colorbar(mappable, cax=cax)
cbar.set_label("Average Peak SWE (mm)", fontsize=14)

y_label_pos = 0.07 if show_nh else -0.02
fig.text(0.5, y_label_pos, "Longitude (°)", ha="center", va="center", fontsize=14)
fig.text(0.09, 0.5, "Latitude (°)", ha="center", va="center", rotation="vertical", fontsize=14)
plt.show()

## Figure 8: mean seasonal cycle per mountain range

SWE per day of the year averaged over each region: the mean over 1980–2020 and the 25th–75th percentile band.

In [ ]:
source_names = 'Hybrid', 'ERA5-Land'  # 'Hybrid', 'ERA5-Land'
sub_regions = 'Pyrenees', 'European Alps', 'Sierra Nevada', 'Canadian Rockies', 'Eastern Sayan', 'Himalaya'
# 'Pyrenees', 'European Alps', 'Sierra Nevada', 'Canadian Rockies', 'Coast Mountains',
# 'Caucasus Mountains', 'Eastern Sayan', 'Pamir Mountains', 'Southern Scandes', 'Himalaya'

In [ ]:
from matplotlib.gridspec import GridSpec

shared_yscale = False  # True: all panels use a common y-axis scale
start_month = 10
month_tick_interval = 2  # Interval for month ticks on the x-axis

n_sub = len(sub_regions)

# Change the font size for the plots
plt.rcParams.update({'font.size': 12})

# Shift amount to start at the specified start date
shift_idx = (
    pd.Timestamp(f'2000-{start_month:02d}-01')
    - pd.Timestamp('2000-01-01')
).days

# Month tick labels
month_starts = pd.date_range(
    start=f'2000-{start_month:02d}-01',
    periods=12,
    freq='MS'
)[::month_tick_interval]

month_labels = [date.strftime('%b') for date in month_starts]

base_date = pd.Timestamp(f'2000-{start_month:02d}-01')

tick_positions = [
    (date - base_date).days + 1
    for date in month_starts
]


def load_doy_dfs(region_name):
    """Load the per-source DOY-SWE dataframes for one region."""
    region_path = region_name.replace(" ", "_").lower()

    dfs = []
    for sname in source_names:
        source_name_path = sname.replace("-", "_").lower()
        filename = f'{source_name_path}_{region_path}_doy_avg_swe.csv'

        df = pd.read_csv(
            local_paths.BASE_PATH / 'results' / 'temp' / filename,
            index_col=0
        )
        dfs.append(df)

    return dfs


def plot_doy_region(ax, region_doy_dfs, show_labels=False):
    """Plot mean and 25th–75th percentile DOY SWE curves for one region."""
    handles_labels = []

    for i, (sname, doy_swe_df) in enumerate(zip(source_names, region_doy_dfs)):
        doy_swe_mean = np.roll(doy_swe_df.mean(axis=0), -shift_idx)
        doy_swe_min = np.roll(doy_swe_df.quantile(0.25, axis=0), -shift_idx)
        doy_swe_max = np.roll(doy_swe_df.quantile(0.75, axis=0), -shift_idx)

        x = range(1, len(doy_swe_mean) + 1)

        h1, = ax.plot(
            x,
            doy_swe_mean,
            color=style[sname]['color'],
            label=f'{sname} Average' if show_labels else None
        )
        h2 = ax.fill_between(
            x,
            doy_swe_min,
            doy_swe_max,
            color=style[sname]['color'],
            alpha=0.2,
            label=f'{sname} 25th–75th Percentile' if show_labels else None
        )

        if show_labels:
            handles_labels.extend([h1, h2])

    ax.set_xlim(1, 365)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(month_labels, rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    return handles_labels


# --- Load all sub-region data ---
sub_doy_dfs = {
    region_name: load_doy_dfs(region_name)
    for region_name in sub_regions
}

# --- Determine a common y-limit only when requested ---
if shared_yscale:
    max_val = max(
        df.quantile(0.75, axis=0).max()
        for region_dfs in sub_doy_dfs.values()
        for df in region_dfs
    )

# --- Layout: one row up to 5 regions; otherwise two rows ---
n_sub_rows = 2 if n_sub > 5 else 1
n_cols = int(np.ceil(n_sub / n_sub_rows))

fig = plt.figure(figsize=(4.5 * n_cols, 3.4 * n_sub_rows))
gs = GridSpec(
    n_sub_rows,
    n_cols,
    figure=fig,
    hspace=0.25,
    wspace=0.25
)

# legend_handles_labels = None
first_ax = None

# --- Sub-region panels ---
for i, region_name in enumerate(sub_regions):
    row = i // n_cols
    col = i % n_cols

    # All panels share the x-axis with the first panel.
    # sharey is deliberately not used: y sharing remains controlled by shared_yscale.
    if first_ax is None:
        ax = fig.add_subplot(gs[row, col])
        first_ax = ax
    else:
        ax = fig.add_subplot(gs[row, col], sharex=first_ax)

    handles = plot_doy_region(
        ax,
        sub_doy_dfs[region_name],
        show_labels=(i == 0)
    )

    # if i == 0:
    #     legend_handles_labels = handles

    # Add subplot label (a, b, c, ...) in the top-left corner of each subplot
    ax.text(
        0.02,
        0.96,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left",
    )

    ax.set_title(region_name)
    ax.set_ylim(bottom=0)

    if shared_yscale:
        ax.set_ylim(top=max_val * 1.05)

        # Avoid repeated tick labels when all panels have the same scale
        if col > 0:
            ax.tick_params(axis='y', labelleft=False)

    # Only show month labels on the bottom row when using two rows
    if row < n_sub_rows - 1:
        ax.tick_params(axis='x', labelbottom=False)

# Create legend manually, only distinguishing between the two sources but skipping the filled percentile bands to avoid clutter
legend_handles_labels = [
    plt.Line2D([0], [0], color=style[sname]['color'], lw=2, label=style[sname]['label'])
    for sname in source_names
]

# Add the legend to the first subplot
legend = first_ax.legend(
    handles=legend_handles_labels,
    loc='upper right',
    # alignment='right',
    # markerfirst=False,
    fontsize=12,
    frameon=False
)

# --- Figure-level axis labels ---
fig.supxlabel("Day of the Year", y=0.01)
fig.supylabel("SWE (mm)", x=0.07)

# Optional legend to the right of the figure, outside the subplots
# fig.legend(
#     handles=legend_handles_labels,
#     loc='center right',
#     bbox_to_anchor=(1.02, 0.5),
#     fontsize=12,
# )

plt.show()

## Figure 9: long-term anomalies over the Northern Hemisphere

Anomalies of the NH-averaged peak SWE and core snow season (CSS) length with respect to their 1980–2020 mean, with the linear trend of HySWE.

In [ ]:
# Load the yearly Northern Hemisphere averages of each variable and source
var_names = ['Peak SWE', 'CSS Length']
source_names = ['Hybrid', 'ERA5-Land']
region = 'Northern Hemisphere'

region_path = region.replace(" ", "_").lower()
dfs_by_var = {var_name: {} for var_name in var_names}
for var_name in var_names:
    var_name_path = var_name.replace(" ", "_").lower()
    for sname in source_names:
        source_name_path = sname.replace("-", "_").lower()
        filename = f'{source_name_path}_{region_path}_yearly_{var_name_path}.csv'
        dfs_by_var[var_name][sname] = pd.read_csv(local_paths.BASE_PATH / 'results' / 'temp' / filename, index_col=0, parse_dates=[0])

In [ ]:
relative_anomaly = False

In [ ]:
from scipy.stats import linregress

# Increase font size for the plot
plt.rcParams.update({"font.size": 16})

# Example:
# source_names = ["Hybrid", "ERA5-Land"]
# var_names = ["Peak SWE", "Core Snow Season Length"]
#
# Expected structure:
# dfs_by_var = {
#     var_name: {
#         source_name: dataframe_with_avg_column,
#         ...
#     },
#     ...
# }

n_vars = len(var_names)

fig, axes = plt.subplots(
    n_vars,
    1,
    figsize=(14, 3.8 * n_vars),
    sharex=True,
    squeeze=False,
)
axes = axes.flatten()

for ax, var_name in zip(axes, var_names):

    units = "%" if relative_anomaly else ("mm" if "SWE" in var_name else "days")

    # Keep track of the temporal coverage in this panel
    all_years = []

    for sname in source_names:

        # Retrieve the DataFrame for this variable and source
        src_df = dfs_by_var[var_name][sname]

        # Extract and prepare the time series
        ts = src_df["avg"].copy()
        ts.index = pd.to_datetime(ts.index)
        ts = ts.sort_index().dropna()

        years = ts.index.year
        all_years.extend(years)

        # Calculate anomaly relative to this source's climatological mean
        anomaly = ts - ts.mean()

        if relative_anomaly:
            anomaly = anomaly / ts.mean() * 100.0

        # Plot anomaly line for every source
        ax.plot(
            years,
            anomaly.values,
            label=style[sname]['label'],
            color=style[sname]['color'],
            linewidth=2,
            zorder=3,
        )

        # Linear trend of the anomalies: printed for both sources, drawn and annotated for HySWE
        slope, intercept, r_value, p_value, std_err = linregress(
            years,
            anomaly.values,
        )
        
        trend = intercept + slope * years
        
        print(
            f"Trend for {var_name} — {sname}: "
            f"{slope * 10:.3f} {units}/decade, "
            f"p-value: {p_value:.2e}, "
            f"R²: {r_value**2:.3f}"
        )

        if sname == "Hybrid":
            ax.plot(
                years,
                trend,
                color=style[sname]['color'],
                linestyle="--",
                linewidth=2.5,
                label=f"Linear trend (HySWE)",
                zorder=4,
            )
            # Add the slope and p-value as text on the plot
            p_value_text = f"p = {p_value:.2f}" if p_value > 1e-3  else "p < 0.001"
            units_text = units + r" $\mathregular{decade^{-1}}$"
            ax.text(
                0.025,
                0.1,
                f"Trend: {slope * 10:.1f} {units_text}\n{p_value_text}",
                transform=ax.transAxes,
                fontsize=16,
                fontweight="normal",
                va="bottom",
                ha="left",
                color=style[sname]['color'],
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'),
                zorder=6,
            )

    # Add a label for the panel letter (a, b, c, ...) in the top-left corner of each subplot
    ax.text(
        0.01,
        0.95,
        panel_labels[var_names.index(var_name)],
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="top",
        ha="left",
        zorder=5,
    )

    # Reference line: climatological mean anomaly
    ax.axhline(0, color="black", linewidth=1, linestyle="--", zorder=2)

    ax.set_ylabel(f"{var_name} Anomaly ({units})")
    # ax.set_title(var_name, loc="left", fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.yaxis.set_major_locator(plt.MaxNLocator(5))
    if ax.get_subplotspec().is_first_row():
        ax.legend(loc="upper right", frameon=False)

    # Set panel-specific temporal coverage
    if all_years:
        ax.set_xlim(min(all_years), max(all_years))

axes[-1].set_xlabel("Year", fontsize=18)

plt.tight_layout()
plt.show()


# Correlation between the two sources for each variable
# This computes correlation over the common available timestamps.
for var_name in var_names:
    if {"Hybrid", "ERA5-Land"}.issubset(source_names):

        hybrid = dfs_by_var[var_name]["Hybrid"]["avg"].copy()
        era5_land = dfs_by_var[var_name]["ERA5-Land"]["avg"].copy()

        hybrid.index = pd.to_datetime(hybrid.index)
        era5_land.index = pd.to_datetime(era5_land.index)

        comparison = pd.concat(
            [
                hybrid.rename("Hybrid"),
                era5_land.rename("ERA5-Land"),
            ],
            axis=1,
        ).dropna()

        correlation = comparison["Hybrid"].corr(comparison["ERA5-Land"])

        print(
            f"Correlation for {var_name} "
            f"(Hybrid vs ERA5-Land): {correlation:.4f}"
        )